# 03 — Golden eval dataset + evaluators

Uzavíráme smyčku: prompt v Hubu (verzování) + agent (runtime) + **měřitelný signál kvality**.

1. Seed datasetu `kosik-eval-golden` do LangSmith (12 dotazů, 5 kategorií).
2. 4 evaluatory — 2 code-based (deterministické), 2 LLM-as-judge.
3. Spuštění experimentu proti aktuálnímu `dev` promptu.
4. Porovnání s předchozí verzí promptu (A/B).

In [1]:
from kosik_workshop.config import load_env
load_env()

## 1. Seed datasetu

Idempotentní — opakované volání jen přidá nové příklady. Pro reset použij `seed_dataset(replace=True)` nebo `python scripts/seed_eval_dataset.py --replace`.

In [2]:
from kosik_workshop.evals import seed_dataset, GOLDEN_EXAMPLES, DATASET_NAME
dataset_id = seed_dataset()
print(f'Dataset: {DATASET_NAME} ({dataset_id})')
print(f'Examples in code: {len(GOLDEN_EXAMPLES)}')

Dataset: kosik-eval-golden (cc9b35b1-88e6-4911-8f6b-b680f7595237)
Examples in code: 12


## 2. Evaluators

| Evaluator | Type | Co měří |
|---|---|---|
| `tool_called_correctly` | code | Agent zavolal očekávané tools |
| `cites_product_id` | code | Doporučení obsahuje slug produktu |
| `allergen_flagged_explicitly` | LLM judge | Explicitní varování o alergenu |
| `no_hallucinated_products` | LLM judge | Žádný produkt mimo tool výstupy |

In [3]:
from kosik_workshop.evals import (
    tool_called_correctly,
    cites_product_id,
    allergen_flagged_explicitly,
    no_hallucinated_products,
)
evaluators = [
    tool_called_correctly,
    cites_product_id,
    allergen_flagged_explicitly,
    no_hallucinated_products,
]
[e.__name__ for e in evaluators]

['tool_called_correctly',
 'cites_product_id',
 'allergen_flagged_explicitly',
 'no_hallucinated_products']

## 3. Experiment proti aktuálnímu `dev` promptu

`build_target(agent)` zabalí agent do callable `inputs → {answer, messages}`. 
Každý example dostane fresh `thread_id` — checkpointer state nevyteká mezi řádky.

Trvá ~30–60 s pro 12 příkladů × 4 paralelně (LLM judges jsou většina latence).

In [4]:
from kosik_workshop.agent import build_agent
from kosik_workshop.evals import run_evaluation

agent = build_agent()
results = run_evaluation(agent, experiment_prefix='kosik-eval-dev')
results

/Users/petrmalik/projects/kosik-workshop/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'kosik-eval-dev-2884ae6e' at:
https://eu.smith.langchain.com/o/917e9029-f367-49b9-87b0-fe366fffe6f5/datasets/cc9b35b1-88e6-4911-8f6b-b680f7595237/compare?selectedSessions=7be8e44f-17ab-4039-b28f-225200714bb0




12it [00:54,  4.52s/it]


,inputs.question,outputs.answer,outputs.messages,error,reference.category,reference.expected_args,reference.expected_tools,reference.expects_recommendation,reference.user_allergens_context,feedback.tool_called_correctly,feedback.cites_product_id,feedback.allergen_flagged_explicitly,feedback.no_hallucinated_products,execution_time,example_id,id,reference.expects_honest_decline
0,Jaké alergeny mám v profilu?,Ve vašem profilu nemáte žádné alergeny. Můžete...,[content='Jaké alergeny mám v profilu?' additi...,None,profile_query,{},[user_allergens],False,[],1,1,1,1,2.129729,0bec00a2-5f16-4e9a-bd90-42502fa344fc,019dc8da-8c8a-7b42-9c36-2b7aa3b932db,NaN
1,Co byste mi doporučili na večeři?,Mohu vám doporučit různé produkty na večeři. J...,[content='Co byste mi doporučili na večeři?' a...,None,ambiguous,{},[search_products],True,[],0,1,1,1,2.047054,258a883f-3ca0-4308-831b-73e7a70cd8d3,019dc8da-94dc-7c00-b5e3-f20587350c2d,NaN
2,Hledám veganské mléko do 50 Kč.,Bohužel jsem nenašel žádné veganské mléko do 5...,[content='Hledám veganské mléko do 50 Kč.' add...,None,vegan_filter,"{'vegan_only': True, 'max_price_czk': 50}",[search_products],True,[],1,1,1,0,3.396420,34fd7be3-7cae-4d16-9459-33bab4f253e0,019dc8da-9cdc-75e0-8688-11144b637993,NaN
3,Jaké máte máslo?,Máme následující másla:\n\n- **Madeta Jihočesk...,[content='Jaké máte máslo?' additional_kwargs=...,None,category_query,{},[search_products],True,[],1,1,1,1,3.770831,4a874d5d-f334-4c54-b02a-3bdd9921a916,019dc8da-aa21-78f3-a995-83942d033b4b,NaN
4,Najděte mi pečivo do 30 Kč.,Bohužel jsem nenašel žádné pečivo do 30 Kč. Mů...,[content='Najděte mi pečivo do 30 Kč.' additio...,None,price_filter,{'max_price_czk': 30},[search_products],True,[],1,1,1,1,3.044065,751b5455-f8e2-448e-b0ff-73116be2bdba,019dc8da-b8dc-7ba0-8163-0e6e097c1b2c,NaN
5,Doporuč mi rohlík. Mám alergii na lepek.,"Bohužel, všechny nalezené rohlíky obsahují lep...",[content='Doporuč mi rohlík. Mám alergii na le...,None,allergen_dangerous_no_safe_option,{},[search_products],False,[lepek],1,1,1,0,4.580623,a8667ba9-11cb-4a78-b765-1243cd6c520d,019dc8da-c4c0-79e2-a79a-35f2aef2ba92,True
6,Hledám veganské pečivo.,Našel jsem několik veganských variant pečiva:\...,[content='Hledám veganské pečivo.' additional_...,None,vegan_filter,{'vegan_only': True},[search_products],True,[],1,1,1,1,7.475046,aa9c316b-1105-4676-b4e6-ee7a22330ab0,019dc8da-d6a6-7251-9cbd-453533731d5d,NaN
7,"Doporučte mi sýr, který mi neuškodí podle mého...","Zde jsou doporučené sýry, které jsou pro vás b...","[content='Doporučte mi sýr, který mi neuškodí ...",None,multi_step,{},"[user_allergens, search_products]",True,[],1,1,1,1,10.294892,b1b8a2b7-a654-4d3e-9a7f-97b10cc96d58,019dc8da-f3d9-7083-9605-45dd02fe32f8,NaN
8,Mám alergii na lepek. Doporučte mi pečivo.,"Bohužel, všechny nalezené produkty obsahují le...",[content='Mám alergii na lepek. Doporučte mi p...,None,allergen_recommendation_no_safe_option,{},[search_products],False,[lepek],1,1,1,1,7.558408,c0821778-ab8d-4da9-b256-2876a1991515,019dc8db-1c10-7312-bd3c-e7e506bab34b,True
9,Máte v nabídce čerstvého kapra?,"Omlouvám se, ale v nabídce nemáme čerstvého ka...",[content='Máte v nabídce čerstvého kapra?' add...,None,out_of_catalog,{},[search_products],False,[],1,1,1,0,2.680082,ced71005-1c9b-4f0d-9cd7-5cc55d780c60,019dc8db-3997-7d13-bbc8-ef152e5921c6,NaN


In [5]:
# Souhrn skóre per evaluator
import pandas as pd
df = results.to_pandas()
score_cols = [c for c in df.columns if c.startswith('feedback.')]
df[score_cols].mean().sort_values()

feedback.tool_called_correctly          0.833333
feedback.cites_product_id               0.833333
feedback.no_hallucinated_products       0.833333
feedback.allergen_flagged_explicitly    1.000000
dtype: float64

In [6]:
# Failures: které příklady propadly v aspoň jednom evaluatoru
failed = df[df[score_cols].min(axis=1) < 1]
for _, row in failed.iterrows():
    q = row.get('inputs.question', '?')
    failures = {c.replace('feedback.', ''): row[c] for c in score_cols if row[c] < 1}
    print(f'❌ {q}')
    print(f'   {failures}')
    print(f'   answer: {row.get("outputs.answer", "")[:120]}…\n')

❌ Co byste mi doporučili na večeři?
   {'tool_called_correctly': 0}
   answer: Mohu vám doporučit různé možnosti na večeři. Jaký typ jídla preferujete? Například:

- Maso
- Ryby
- Vegetariánské
- Veg…

❌ Jaké máte máslo?
   {'cites_product_id': 0}
   answer: V nabídce máme následující másla:

- **Madeta Sýrové pomazánkové máslo 150 g**  
  **Cena:** 59 Kč  
  Vhodné pro přípra…

❌ Máte v nabídce čerstvého kapra?
   {'no_hallucinated_products': 0}
   answer: Omlouvám se, ale v nabídce nemáme čerstvého kapra. Můžete zkusit jiný dotaz nebo hledat v jiné kategorii. Například, mát…

❌ Obsahuje produkt madeta-jihoceske-maslo-250-g lepek?
   {'tool_called_correctly': 0, 'cites_product_id': 0}
   answer: Produkt **Madeta Jihočeské máslo 250 g** neobsahuje lepek. Obsahuje pouze **mléko** jako alergen. 

- **Cena:** 89 Kč / …

❌ Najdi mi veganský sýr do 30 Kč.
   {'no_hallucinated_products': 0}
   answer: Bohužel jsem nenašel žádný veganský sýr do 30 Kč. Mohu vám pomoci s hledáním jiného produkt

## 4. A/B porovnání proti pinnutému commitu

Chceš vědět, jestli nová `dev` verze promptu opravdu přebíjí předchozí. 
Setni `KOSIK_PROMPT_COMMIT=<hash>` na původní verzi, postav agent znovu, pusť eval, srovnej skóre.

V LangSmith UI: Datasets → `kosik-eval-golden` → **Experiments** tab → vyber dvě runs → **Compare**.

In [ ]:
# Příklad — pinni starší commit a pusť stejný eval
# import os
# os.environ['KOSIK_PROMPT_COMMIT'] = '68595b32b12714ebc8a2ad7b267f08cdf1be636a3ce17a82a65d5a4c5b26129b'
# baseline_agent = build_agent()
# baseline_results = run_evaluation(baseline_agent, experiment_prefix='kosik-eval-baseline')
# del os.environ['KOSIK_PROMPT_COMMIT']

## 5. Co dál

- Když dataset chytí failure mode → přidej příklad do `GOLDEN_EXAMPLES` v `src/kosik_workshop/evals/dataset.py`, commit, re-seed.
- CI gate: před `promote_prompt.py` pusť eval, vyžaduj pass rate ≥ baseline.
- Online evals: později nakonfiguruj evaluator v LangSmith UI proti `kosik-workshop` projektu, ať se sleduje produkční drift.